# Assignment: The Visualizations Continue

Every CSV file is stored in the same folder as this notebook and is loaded with a relative path, for example `pd.read_csv('2010_songs.csv')`. To run the notebook, unzip the folder and open the notebook from inside it.

Files in the folder:
- `The_Visualizations_Continue.ipynb` (this notebook)
- `2010_songs.csv` (Part I)
- `GDP_annual_growth.csv` (Part II)
- `google_apps.csv` (Part III)
- `youtube.csv` (Part IV)

Required libraries: pandas, NumPy, Bokeh, ipywidgets, and jupyter_bokeh (the jupyter_bokeh package lets Bokeh plots and widgets display inside JupyterLab).

In [1]:
import math
import numpy as np
import pandas as pd

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import Range1d, LinearColorMapper, ColorBar, HoverTool, ColumnDataSource
from bokeh.layouts import gridplot
from bokeh.palettes import Viridis256, Plasma256

import ipywidgets as widgets
from IPython.display import display

output_notebook()


def style_plot(p):
    """Apply readable font sizes to a Bokeh figure (title, axis labels, and tick labels)."""
    p.title.text_font_size = '16pt'
    p.xaxis.axis_label_text_font_size = '13pt'
    p.yaxis.axis_label_text_font_size = '13pt'
    p.xaxis.major_label_text_font_size = '11pt'
    p.yaxis.major_label_text_font_size = '11pt'
    return p

Loading BokehJS ...

## Part I: Basic Plotting Through Bokeh

### Loading the songs data

In [2]:
songs = pd.read_csv('2010_songs.csv', encoding='utf-8-sig')   # relative path
print(songs.shape)
print('Songs are listed in order of popularity:', songs['pop'].is_monotonic_decreasing)
songs[['title', 'artist', 'year', 'energy', 'danceability', 'pop']].head()

(100, 14)
Songs are listed in order of popularity: True


,title,artist,year,energy,danceability,pop
0,bad guy,Billie Eilish,2019,43,70,94
1,7 rings,Ariana Grande,2019,32,78,90
2,Old Town Road - Remix,Lil Nas X,2019,62,88,89
3,SeÃ±orita,Shawn Mendes,2019,55,76,88
4,rockstar (feat. 21 Savage),Post Malone,2018,52,59,88


### Scatter plot: energy by index

In [3]:
p1 = figure(title='Energy Level of Popular Songs from the 2010s',
            x_axis_label='Index (Song Position in the Data Set)',
            y_axis_label='Energy Level',
            width=850, height=450, y_range=Range1d(10, 100))

# Orange triangles
p1.scatter(x=songs.index, y=songs['energy'], marker='triangle',
           size=12, color='orange')

# Center the title and make both axis labels bold
p1.title.align = 'center'
p1.xaxis.axis_label_text_font_style = 'bold'
p1.yaxis.axis_label_text_font_style = 'bold'

style_plot(p1)
show(p1)

### Line plot: energy and danceability by index

In [4]:
p2 = figure(title='Energy Level vs. Danceability of Popular Songs from the 2010s',
            x_axis_label='Index (Song Position in the Data Set)',
            y_axis_label='Score (0–100)',
            width=850, height=450, y_range=Range1d(0, 100))

p2.line(x=songs.index, y=songs['energy'], line_width=3, color='darkorange',
        legend_label='Energy', muted_color='darkorange', muted_alpha=0.15)
p2.line(x=songs.index, y=songs['danceability'], line_width=3, color='royalblue',
        legend_label='Danceability', muted_color='royalblue', muted_alpha=0.15)

# Legend in the lower right with an azure background; clicking a legend item mutes that line
p2.legend.location = 'bottom_right'
p2.legend.background_fill_color = 'azure'
p2.legend.click_policy = 'mute'
p2.legend.label_text_font_size = '12pt'

p2.title.align = 'center'
style_plot(p2)
show(p2)

In [5]:
# Supporting numbers for the takeaways
print(songs[['energy', 'danceability']].describe().round(1))
print()
print('Correlation between energy and danceability:',
      round(songs['energy'].corr(songs['danceability']), 2))
print()
print('Mean of the first 50 songs (most popular):')
print(songs[['energy', 'danceability']].iloc[:50].mean().round(1))
print('Mean of the last 50 songs:')
print(songs[['energy', 'danceability']].iloc[50:].mean().round(1))
print()
print('Lowest energy songs:')
print(songs.nsmallest(3, 'energy')[['title', 'artist', 'energy', 'danceability']].to_string())

       energy  danceability
count   100.0         100.0
mean     68.0          65.4
std      16.3          11.9
min      17.0          21.0
25%      56.8          58.0
50%      68.0          67.0
75%      80.0          74.0
max      95.0          91.0

Correlation between energy and danceability: 0.06

Mean of the first 50 songs (most popular):
energy          62.6
danceability    65.8
dtype: float64
Mean of the last 50 songs:
energy          73.3
danceability    65.0
dtype: float64

Lowest energy songs:
                                       title         artist  energy  danceability
19                         Dancing On My Own    Calum Scott      17            68
30  i hate u i love u (feat. olivia o'brien)          gnash      28            49
39                        Little Do You Know  Alex & Sierra      29            50


**Study the visualization and provide a couple takeaways.**

- **Energy changes much more from song to song than danceability does.** The orange energy line has sharp peaks and dips, ranging from 17 to 95, while the blue danceability line stays in a narrower band (standard deviation of about 11.9 compared with about 16.3 for energy). Most hit songs from this decade had a danceability score between 50 and 90, which suggests a steady, danceable beat was common among popular songs regardless of their energy level.
- **The two measures are almost unrelated.** When one line rises, the other line does not consistently follow. The correlation between energy and danceability is only about 0.06. For example, "She Looks So Perfect" has very high energy (95) but below-average danceability (49), and "Dancing On My Own" has low energy but solid danceability (68).
- **The most popular songs tend to have lower energy.** The songs are listed in order of popularity, so the left side of the chart holds the most popular songs. The first 50 songs average about 62.6 for energy, while the last 50 average about 73.3. Danceability stays about the same across both halves (about 65). Several top songs, such as "bad guy" and "7 rings," are relatively calm but still very danceable.
- **Muting is useful for this chart.** Because the two lines cross each other many times, clicking a legend item to mute one line makes the other line much easier to follow.

## Part II: More Advanced Bokeh Plotting

### Loading the GDP data and calculating the median

Each row is a country and each year from 1961 to 2023 is a column. To find the "typical" growth for each year, I use `np.nanmedian()` down each year column. This function ignores missing values (NaN), which matters because some countries do not have data for every year.

In [6]:
gdp = pd.read_csv('GDP_annual_growth.csv')   # relative path
print(gdp.shape)
print('Missing values:', gdp.isna().sum().sum())

year_columns = gdp.columns[2:]
years = year_columns.astype(int)

us_growth = gdp.loc[gdp['Country Code'] == 'USA', year_columns].iloc[0].astype(float).values
median_growth = np.nanmedian(gdp[year_columns].values.astype(float), axis=0)

growth = pd.DataFrame({'Year': years, 'US': us_growth, 'Median': median_growth})
growth.round(2).head()

(203, 65)
Missing values: 345


,Year,US,Median
0,1961,0.62,2.28
1,1962,4.48,2.15
2,1963,2.91,2.46
3,1964,4.34,3.01
4,1965,5.08,2.44


In [7]:
# Values used to set a y-axis range that shows every data point for both lines
print('Lowest value :', round(growth[['US', 'Median']].min().min(), 2))
print('Highest value:', round(growth[['US', 'Median']].max().max(), 2))

Lowest value : -4.4
Highest value: 6.31


### US vs. median GDP growth on one plot

In [8]:
p3 = figure(title='Annual GDP Growth: United States vs. Median of All Countries (1961–2023)',
            x_axis_label='Year', y_axis_label='Annual GDP Growth (%)',
            width=950, height=500,
            y_range=Range1d(-6, 8))   # manual min and max so no points are cut off

# US: solid grey line
p3.line(growth['Year'], growth['US'], line_width=3, color='grey',
        legend_label='United States')

# Median: red line with circles that have a red boundary and a white interior
p3.line(growth['Year'], growth['Median'], line_width=2, color='red',
        legend_label='Median (All Countries)')
p3.scatter(growth['Year'], growth['Median'], marker='circle', size=7,
           line_color='red', fill_color='white', line_width=2,
           legend_label='Median (All Countries)')

# Clicking a legend item hides that line
p3.legend.click_policy = 'hide'
p3.legend.location = 'bottom_left'   # keeps the legend away from the 2021 rebound
p3.legend.label_text_font_size = '12pt'

# No grid lines
p3.xgrid.grid_line_color = None
p3.ygrid.grid_line_color = None

style_plot(p3)
show(p3)

In [9]:
# Supporting numbers for the conclusions
growth['US Above Median'] = growth['US'] > growth['Median']
print('Years the US grew faster than the median:', growth['US Above Median'].sum(), 'of', len(growth))
for start, end in [(1961, 1979), (1980, 1999), (2000, 2023)]:
    period = growth[(growth['Year'] >= start) & (growth['Year'] <= end)]
    print(f"{start}-{end}: US above median in {period['US Above Median'].sum()} of {len(period)} years | "
          f"US average {period['US'].mean():.2f}% | median average {period['Median'].mean():.2f}%")
print()
print('Standard deviation - US:', round(growth['US'].std(), 2), ' Median:', round(growth['Median'].std(), 2))
print('Years with negative US growth:', growth.loc[growth['US'] < 0, 'Year'].tolist())
print('Years with negative median growth:', growth.loc[growth['Median'] < 0, 'Year'].tolist())

Years the US grew faster than the median: 32 of 63
1961-1979: US above median in 11 of 19 years | US average 2.71% | median average 2.41%
1980-1999: US above median in 15 of 20 years | US average 2.07% | median average 1.30%
2000-2023: US above median in 6 of 24 years | US average 1.41% | median average 1.98%

Standard deviation - US: 2.06  Median: 1.29
Years with negative US growth: [1970, 1974, 1975, 1980, 1982, 1991, 2001, 2008, 2009, 2020]
Years with negative median growth: [1982, 2009, 2020]


**Draw a couple conclusions from the visualization.**

- **The United States swings more than the typical country.** The grey US line has much bigger peaks and dips than the red median line (standard deviation of about 2.06 compared with about 1.29). The US had negative growth in 10 of the 63 years, including 1974–1975, 1980, 1982, 1991, 2008–2009, and 2020. The median only went negative three times: 1982, 2009, and 2020.
- **Global shocks show up in both lines.** The 1982 recession, the 2009 financial crisis, and the 2020 COVID-19 pandemic caused both lines to drop below zero at the same time. 2020 was the deepest drop for the median (about −4.4%), and both lines then rebounded sharply in 2021.
- **The US has fallen behind the typical country since 2000.** From 1980 to 1999, US growth was above the median in 15 of 20 years. From 2000 to 2023, it was above the median in only 6 of 24 years, averaging about 1.41% compared with a median of about 1.98%. This is consistent with a large, mature economy growing more slowly than many developing economies.
- **Data note:** Some countries in this file appear to have the same value repeated for many early years (for example, Afghanistan from 1961 to 2001). Those repeated values probably fill in missing data, so the early median should be read with some caution.

### Gridplot: horizontal arrangement

Both plots in each gridplot share the same `Range1d` object for the y-axis (from −6 to 8), so the scale is identical and every data point is visible. Each gridplot gets its own new `Range1d`, because Bokeh does not allow one model to be shown in two separate layouts.

In [10]:
def make_gdp_plot(values, title, color, y_range):
    p = figure(title=title, x_axis_label='Year', y_axis_label='Annual GDP Growth (%)',
               width=560, height=420, y_range=y_range)
    p.line(growth['Year'], values, line_width=3, color=color)
    p.scatter(growth['Year'], values, marker='circle', size=5, color=color)
    return style_plot(p)

horizontal_y_range = Range1d(-6, 8)   # one shared range for both plots
us_plot = make_gdp_plot(growth['US'], 'United States GDP Growth', 'magenta', horizontal_y_range)
median_plot = make_gdp_plot(growth['Median'], 'Median GDP Growth (All Countries)', 'darkmagenta', horizontal_y_range)

# Horizontal arrangement: one row, two columns
show(gridplot([[us_plot, median_plot]]))

### Gridplot: vertical arrangement

In [11]:
vertical_y_range = Range1d(-6, 8)   # one shared range for both plots
us_plot_v = make_gdp_plot(growth['US'], 'United States GDP Growth', 'magenta', vertical_y_range)
median_plot_v = make_gdp_plot(growth['Median'], 'Median GDP Growth (All Countries)', 'darkmagenta', vertical_y_range)

for p in (us_plot_v, median_plot_v):
    p.width = 950
    p.height = 350

# Vertical arrangement: two rows, one column
show(gridplot([[us_plot_v], [median_plot_v]]))

**Out of the three arrangements you created for this data, which one do you prefer? Please explain.**

I prefer the **first arrangement, with both lines on the same plot**. The goal of this visualization is to compare US growth with the typical country, and that comparison is easiest when the two lines sit on top of each other. It is immediately clear in which years the US was above or below the median, where the lines cross, and that the 1982, 2009, and 2020 drops happened at the same time. The hide click policy also lets me remove one line when I want to study the other on its own.

Of the two gridplots, the **vertical arrangement** works better than the horizontal one. Because the plots are stacked, the years line up directly on top of each other, so I can trace a single year (such as 2009) straight down from one plot to the other. In the horizontal arrangement, my eyes have to move back and forth across the page and match up years by position, which is harder. The separate plots are still useful when each line needs to be studied without the other one getting in the way.

## Part III: Playing Around With Different Variations

### Loading the apps data

In [12]:
apps = pd.read_csv('google_apps.csv', encoding='utf-8-sig')   # relative path
print(apps.shape)
print('Number of categories:', apps['Category'].nunique())
print('Apps with a rating of 0 (no ratings yet):', (apps['Rating'] == 0).sum())
apps.head()

(200, 8)
Number of categories: 20
Apps with a rating of 0 (no ratings yet): 90


,App Name,Category,Rating,Minimum Installs,Maximum Installs,Size,Developer Id,Content Rating
0,Ninja Well,Action,0.0,1,3,12M,Culp Rewards.,Everyone
1,Jelly Man Climb,Action,0.0,1000,1422,50M,"in Motion, Inc.",Everyone
2,Stickman Spider Rope Hero Gangstar Crime,Action,3.2,50000,53995,12M,Rowan Lagan,Everyone 10+
3,Running Stick,Action,4.0,10000,34691,19M,ZPLAY Games,Everyone
4,Siren Head Rope hero Vs Super Heros,Action,4.0,1000,2692,64M,Dev3a,Everyone


### Scatter plot with LinearColorMapper

In [13]:
apps = apps.assign(Index=apps.index)

rating_mapper = LinearColorMapper(palette=Viridis256, low=0, high=5)

p4 = figure(title='Google Play Store App Ratings by Index',
            x_axis_label='Index', y_axis_label='Rating (0–5)',
            width=900, height=450)
p4.scatter(x='Index', y='Rating', source=ColumnDataSource(apps), size=10,
           color={'field': 'Rating', 'transform': rating_mapper})

# Color bar so the reader knows what each color means
p4.add_layout(ColorBar(color_mapper=rating_mapper, title='Rating',
                       major_label_text_font_size='11pt'), 'right')

style_plot(p4)
show(p4)

### Scatter plot with a tooltip

In [14]:
p5 = figure(title='Google Play Store App Ratings (Hover for Details)',
            x_axis_label='Index', y_axis_label='Rating (0–5)',
            width=900, height=450)
p5.scatter(x='Index', y='Rating', source=ColumnDataSource(apps), size=10, color='teal')

p5.add_tools(HoverTool(tooltips=[('App', '@{App Name}'),
                                 ('Category', '@Category'),
                                 ('Rating', '@Rating')]))

style_plot(p5)
show(p5)

In [15]:
# Supporting numbers for the conclusions
rated = apps[apps['Rating'] > 0]
print('Apps with no rating (0):', (apps['Rating'] == 0).sum(), 'of', len(apps))
print('Mean rating, all apps:', round(apps['Rating'].mean(), 2))
print('Mean rating, rated apps only:', round(rated['Rating'].mean(), 2))
print('Share of rated apps at 4.0 or higher:', round((rated['Rating'] >= 4).mean() * 100, 1), '%')
print()
print('Median maximum installs - unrated apps:', apps.loc[apps['Rating'] == 0, 'Maximum Installs'].median())
print('Median maximum installs - rated apps  :', rated['Maximum Installs'].median())
print()
print('Apps are grouped by category in the file (10 per category):')
print(apps['Category'].iloc[::10].tolist())

Apps with no rating (0): 90 of 200
Mean rating, all apps: 2.19
Mean rating, rated apps only: 3.97
Share of rated apps at 4.0 or higher: 62.7 %

Median maximum installs - unrated apps: 106.5
Median maximum installs - rated apps  : 6039.0

Apps are grouped by category in the file (10 per category):
['Action', 'Adventure', 'Arcade', 'Art & Design', 'Auto & Vehicles', 'Beauty', 'Books & Reference', 'Business', 'Communication', 'Dating', 'Education', 'Entertainment', 'Finance', 'Food & Drink', 'Health & Fitness', 'Maps & Navigation', 'Music', 'Photography', 'Social', 'Tools']


**Study this plot and draw a couple conclusions.**

- **Almost half of the apps have a rating of 0.** The row of points along the bottom of the chart shows 90 of the 200 apps sitting at 0. These are mostly small apps with very few installs (a median of about 107 maximum installs, compared with about 6,039 for rated apps), so a 0 here most likely means "not rated yet" rather than "rated poorly." Those apps pull the overall average rating down to about 2.2.
- **Apps that do have ratings are mostly rated well.** Among the 110 rated apps, the average is about 4.0, and about 63% of them are rated 4.0 or higher. The rated points cluster near the top of the chart, so users who leave ratings tend to rate positively.
- **The index follows the category order.** The file lists 10 apps per category in alphabetical order (Action, Adventure, Arcade, and so on). The tooltip makes this easy to confirm, and it shows that some stretches of the x-axis (certain categories) have many more 0 ratings than others. That leads into the grouped plot below.

### Scatter plot: mean rating by category

In [16]:
category_means = (apps.groupby('Category', as_index=False)['Rating'].mean()
                      .sort_values('Rating', ascending=False))
category_means['Rating'] = category_means['Rating'].round(2)
category_source = ColumnDataSource(category_means)

mean_mapper = LinearColorMapper(palette=Plasma256,
                                low=category_means['Rating'].min(),
                                high=category_means['Rating'].max())

p6 = figure(title='Mean Google Play Store Rating by App Category',
            x_range=category_means['Category'].tolist(),
            x_axis_label='App Category', y_axis_label='Mean Rating (0–5)',
            width=1000, height=550, y_range=Range1d(0, 4))
p6.scatter(x='Category', y='Rating', source=category_source, size=17,
           color={'field': 'Rating', 'transform': mean_mapper},
           line_color='black')

# Rotate the category labels using a numeric value (radians); 0.9 is about 52 degrees
p6.xaxis.major_label_orientation = 0.9

p6.add_layout(ColorBar(color_mapper=mean_mapper, title='Mean Rating',
                       major_label_text_font_size='11pt'), 'right')
p6.add_tools(HoverTool(tooltips=[('Category', '@Category'), ('Mean Rating', '@Rating')]))

style_plot(p6)
show(p6)

In [17]:
# Supporting numbers: mean rating, rated-only mean, and number of 0 ratings per category
summary = apps.groupby('Category')['Rating'].agg(
    Mean_Rating='mean',
    Zero_Ratings=lambda r: (r == 0).sum(),
    Rated_Only_Mean=lambda r: r[r > 0].mean())
print(summary.round(2).sort_values('Mean_Rating', ascending=False))

                   Mean_Rating  Zero_Ratings  Rated_Only_Mean
Category                                                     
Photography               3.43             1             3.81
Music                     3.38             2             4.22
Social                    2.98             3             4.26
Tools                     2.94             2             3.68
Auto & Vehicles           2.86             3             4.09
Action                    2.83             3             4.04
Arcade                    2.62             4             4.37
Art & Design              2.60             2             3.25
Dating                    2.54             4             4.23
Maps & Navigation         2.41             3             3.44
Adventure                 2.40             4             4.00
Books & Reference         2.12             5             4.24
Finance                   2.02             5             4.04
Communication             1.82             6             4.55
Food & D

**What are some conclusions that can be drawn from this grouped visualization?**

- **Photography and Music have the highest mean ratings** (about 3.43 and 3.38), shown as the bright yellow dots on the left. **Health and Fitness has the lowest** (about 0.61), followed by Business (about 0.96) and Entertainment (about 1.08), shown as the dark purple dots on the right.
- **The 0 ratings drive most of the differences between categories.** Health and Fitness and Business each have 8 apps out of 10 with a rating of 0, while Photography has only 1. A low category mean in this chart mostly means that many apps in that category have not been rated yet, not that users dislike them. When the 0 values are removed, every category has a mean between about 3.0 and 4.8. Business, for example, jumps from the second lowest mean (0.96) to the highest (4.80), because its two rated apps are both rated very well.
- **Every category has a small sample.** With only 10 apps per category, one or two apps can move a category's mean a lot. These results describe this sample of 200 apps, not the whole Google Play Store. A larger sample, or a version that leaves out unrated apps, would give a fairer comparison.
- **Grouping makes the pattern much clearer.** In the index-based plots, the categories were hard to compare. Averaging by category and sorting from highest to lowest turns 200 scattered points into 20 easy-to-rank values, and the color scale reinforces the ranking.

## Part IV: Active Interactions Through Widgets

### Loading the YouTube data

In [18]:
youtube = pd.read_csv('youtube.csv', encoding='utf-8-sig')   # relative path
print(youtube.shape)
youtube[['Youtuber', 'video count', 'video views billions', 'category']].head()

(940, 6)


,Youtuber,video count,video views billions,category
0,MrBeast,803,52.403,Entertainment
1,T-Series,21237,258.624,Music
2,Cocomelon - Nursery Rhymes,1188,182.881,Education
3,? Kids Diana Show,1230,103.973,People & Blogs
4,Vlad and Niki,730,91.646,Entertainment


### Scatter plot with slider widgets

Two `IntSlider` widgets from ipywidgets control the maximum value of each axis. Setting `continuous_update=False` means the plot only updates when the mouse button is released, not while the slider is being dragged. The y-axis slider uses a vertical orientation and sits to the left of the plot, and the x-axis slider uses a horizontal orientation and sits below the plot.

In [19]:
x_max_value = int(math.ceil(youtube['video count'].max() / 1000) * 1000)        # 80,000
y_max_value = int(math.ceil(youtube['video views billions'].max() / 10) * 10)   # 260

def draw_youtube_plot(x_max, y_max):
    p = figure(title='YouTube Channels: Video Count vs. Video Views',
               x_axis_label='Video Count (Number of Videos Uploaded)',
               y_axis_label='Video Views (in billions)',
               width=850, height=500,
               x_range=Range1d(0, x_max), y_range=Range1d(0, y_max))
    p.scatter(x='video count', y='video views billions', source=ColumnDataSource(youtube),
              size=8, color='crimson', alpha=0.7)

    # Background fill color and a different grid line color
    p.background_fill_color = 'whitesmoke'
    p.xgrid.grid_line_color = 'lightsteelblue'
    p.ygrid.grid_line_color = 'lightsteelblue'

    p.add_tools(HoverTool(tooltips=[('Channel', '@Youtuber'),
                                    ('Videos', '@{video count}{0,0}'),
                                    ('Views (billions)', '@{video views billions}')]))
    style_plot(p)
    show(p)

# Integer sliders that only trigger when the mouse is released (continuous_update=False)
x_slider = widgets.IntSlider(value=x_max_value, min=1000, max=x_max_value, step=1000,
                             description='Max X:', continuous_update=False,
                             orientation='horizontal',
                             layout=widgets.Layout(width='850px'))
y_slider = widgets.IntSlider(value=y_max_value, min=5, max=y_max_value, step=5,
                             description='Max Y:', continuous_update=False,
                             orientation='vertical',
                             layout=widgets.Layout(height='500px'))

plot_output = widgets.interactive_output(draw_youtube_plot, {'x_max': x_slider, 'y_max': y_slider})

# Vertical y-axis slider on the left; plot with the horizontal x-axis slider underneath
display(widgets.HBox([y_slider, widgets.VBox([plot_output, x_slider])]))

In [ ]:
# Supporting numbers for the correlation question
print('Pearson correlation :', round(youtube['video count'].corr(youtube['video views billions']), 2))
print('Spearman correlation:', round(youtube['video count'].corr(youtube['video views billions'], method='spearman'), 2))
print()
print('Most videos uploaded:')
print(youtube.nlargest(4, 'video count')[['Youtuber', 'video count', 'video views billions']].to_string(index=False))
print()
print('Most views:')
print(youtube.nlargest(4, 'video views billions')[['Youtuber', 'video count', 'video views billions']].to_string(index=False))

Pearson correlation : 0.16


**Does there seem to be a strong correlation between these two variables? Please explain.**

No, there is **not a strong correlation** between video count and video views. The points do not follow an upward line. Instead, most channels are packed into the lower-left corner (fewer than about 10,000 videos and fewer than about 20 billion views), with points spread out in two different directions:

- **Many videos, modest views:** Channels such as Ch3Thailand, GMM25Thailand, and MNCTV OFFICIAL have uploaded more than 75,000 videos each but have fewer than 25 billion views. These are mostly TV networks that upload large numbers of clips.
- **Few videos, huge views:** Cocomelon (about 1,200 videos and 183 billion views), Like Nastya (about 890 videos and 101 billion views), and Vlad and Niki (about 730 videos and 92 billion views) get enormous view counts from a relatively small library, mostly because children rewatch the same videos many times.

The correlation numbers back this up. The Pearson correlation is only about 0.16, and the Spearman (rank-based) correlation is about 0.22, both weak. Using the sliders to zoom in on the main cluster (for example, setting the x maximum to 10,000 and the y maximum to 30) still shows a loose cloud with no clear trend. Uploading more videos does not, by itself, lead to more views. The type of content and how often people rewatch it matter much more.